<a href="https://colab.research.google.com/github/attatharva/building-chatgpt-from-scratch/blob/main/chatgpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

uploading the dataset


In [1]:
from google.colab import files
uploaded = files.upload()

Saving input.txt to input.txt


checking if it has been uploaded


In [2]:
!ls

input.txt  sample_data


setting the hyper parameters


In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 20
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

torch.manual_seed(1337)

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


loading the dataset

In [4]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("Dataset size:", len(text))
print(text[:500])

Dataset size: 2194278
THE HUMAN ORGAN SYSTEMS - EXPANDED DATASET


This dataset describes the organ systems of the human body. It explains the location, function, and structure of many organs, and includes questions and answers about each organ to help explain human biology clearly.



------------------------------------------------------------

THE CIRCULATORY SYSTEM

------------------------------------------------------------

The circulatory system is 


seeing the vocabulary size

In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

print("Vocabulary size:", vocab_size)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

Vocabulary size: 59


encode data as train and validation

In [6]:
data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print("Train size:", len(train_data))
print("Validation size:", len(val_data))

Train size: 1974850
Validation size: 219428


data loader functions

In [7]:
def get_batch(split):
    data_source = train_data if split == 'train' else val_data

    ix = torch.randint(len(data_source) - block_size, (batch_size,))

    x = torch.stack([data_source[i:i+block_size] for i in ix])
    y = torch.stack([data_source[i+1:i+block_size+1] for i in ix])

    return x.to(device), y.to(device)

loss function

In [8]:
@torch.no_grad()
def estimate_loss():
    out = {}

    model.eval()

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, Y = get_batch(split)

            logits, loss = model(X, Y)

            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()

    return out

model classes

In [9]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


the model

In [10]:
model = GPTLanguageModel()

m = model.to(device)

print(
    sum(p.numel() for p in m.parameters())/1e6,
    "M parameters"
)

10.784315 M parameters


optimiser

In [11]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

training loop

In [12]:
for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:

        losses = estimate_loss()

        print(
            f"step {iter}: "
            f"train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

step 0: train loss 4.0586, val loss 4.0613
step 500: train loss 0.1503, val loss 1.5939
step 1000: train loss 0.0771, val loss 1.8463
step 1500: train loss 0.0709, val loss 1.8192
step 2000: train loss 0.0674, val loss 1.7357
step 2500: train loss 0.0657, val loss 1.7894
step 3000: train loss 0.0644, val loss 1.8514
step 3500: train loss 0.0636, val loss 1.8971
step 4000: train loss 0.0625, val loss 1.7467
step 4500: train loss 0.0610, val loss 1.8265
step 4999: train loss 0.0618, val loss 1.9138


save model parameters


In [13]:
torch.save(model.state_dict(), "gpt_percy.pth")

In [16]:
from google.colab import files
files.download("gpt_percy.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

generate

In [16]:


prompt = "tell me about cardiac muscle "
context = torch.tensor([encode(prompt)], dtype=torch.long).to(device)

generated = decode(
    m.generate(
        context,
        max_new_tokens=2000
    )[0].tolist()
)

print(generated)

tell me about cardiac muscle is to pump blood throughout the body without tiring. The cardiac muscle is located in the walls of the heart. The cardiac muscle is made of cardiac muscle tissue. The cardiac muscle plays an important role in keeping the whole body functioning well. Without a healthy cardiac muscle, the muscular system would not be able to work properly. The main function of the cardiac muscle is to pump blood throughout the body without tiring. Doctors who study the muscular system pay close attention to the health of the cardiac muscle. Without a healthy cardiac muscle, the muscular system would not be able to work properly. The cardiac muscle is made of cardiac muscle tissue. The cardiac muscle plays an important role in keeping the whole body functioning well. The cardiac muscle is a good example of how organs are specialized for a specific job.

Q: Which system is the cardiac muscle part of?
A: The cardiac muscle is part of the muscular system.

Q: Why is the cardiac m